# Mental Health Score Prediction

This notebook builds a machine learning model to predict **Mental Health Score** from student social-media, lifestyle, academic, and demographic information.

### Workflow
1. Import libraries
2. Load and understand the dataset
3. Perform exploratory data analysis (EDA)
4. Check and clean the data
5. Create features
6. Encode and preprocess features
7. Split the data into training and testing sets
8. Train Linear Regression and Random Forest models
9. Tune the Random Forest model
10. Evaluate and compare the models
11. Save the model

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv("/content/Student Social Media And Mental Health Impact.csv")

df.head()

## 3. Understand the Dataset

In [ ]:
# Number of rows and columns
print("Rows and columns:", df.shape)

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Column names
print(df.columns.tolist())

In [ ]:
# Dataset information and data types
df.info()

In [ ]:
# Summary statistics for numerical columns
df.describe()

## 4. Check Data Quality

In [ ]:
# Check missing values in each column
df.isnull().sum()

In [ ]:
# Check duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

## 5. Exploratory Data Analysis (EDA)

EDA helps us understand the distribution of the target variable and the relationship between important features and Mental Health Score.

In [ ]:
# Distribution of the target variable
plt.figure(figsize=(8, 5))
sns.histplot(df["Mental_Health_Score"], kde=True)
plt.title("Distribution of Mental Health Score")
plt.xlabel("Mental Health Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Correlation between numerical variables
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Stress Level vs Mental Health Score
print(df["Stress_Level"].unique())

stress_order = ["Low", "Medium", "High", "Very High"]

plt.figure(figsize=(8, 5))
sns.boxenplot(
    x="Stress_Level",
    y="Mental_Health_Score",
    data=df,
    order=stress_order
)
plt.title("Stress Level vs Mental Health Score")
plt.show()

In [ ]:
# Daily social-media usage vs Mental Health Score
plt.figure(figsize=(8, 5))
sns.scatterplot(
    x="Avg_Daily_Usage_Hours",
    y="Mental_Health_Score",
    data=df
)
plt.title("Daily Usage Hours vs Mental Health Score")
plt.show()

In [ ]:
# Sleep hours vs Mental Health Score
plt.figure(figsize=(8, 5))
sns.scatterplot(
    x="Sleep_Hours_Per_Night",
    y="Mental_Health_Score",
    data=df
)
plt.title("Sleep Hours vs Mental Health Score")
plt.show()

In [ ]:
# Physical activity hours vs Mental Health Score
plt.figure(figsize=(8, 5))
sns.scatterplot(
    x="Physical_Activity_Hours",
    y="Mental_Health_Score",
    data=df
)
plt.title("Physical Activity Hours vs Mental Health Score")
plt.show()

In [ ]:
# Most-used platform
platform_counts = df["Most_Used_Platform"].value_counts()
print(platform_counts)

plt.figure(figsize=(8, 5))
sns.countplot(
    x="Most_Used_Platform",
    data=df,
    order=platform_counts.index
)
plt.title("Most Used Social Media Platform")
plt.xlabel("Platform")
plt.ylabel("Number of Students")
plt.xticks(rotation=45)
plt.show()

## 6. Check for Outliers

In [ ]:
# Select numerical features
numeric_features = df.select_dtypes(include="number")

# Calculate IQR
Q1 = numeric_features.quantile(0.25)
Q3 = numeric_features.quantile(0.75)
IQR = Q3 - Q1

# Calculate lower and upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
outliers = (
    (numeric_features < lower_bound) |
    (numeric_features > upper_bound)
)

print("Outliers in each numerical column:")
print(outliers.sum())

## 7. Data Cleaning

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()

# Keep Physical Activity Hours at a realistic minimum of 0
df["Physical_Activity_Hours"] = df["Physical_Activity_Hours"].clip(lower=0)

print("Dataset shape after cleaning:", df.shape)

## 8. Check Skewness

Skewness shows whether a numerical feature is approximately symmetric or has a long tail.

- Near 0 → approximately symmetric
- Positive → right-skewed
- Negative → left-skewed

In [ ]:
numeric_columns = df.select_dtypes(include="number")
skewness = numeric_columns.skew()

print(skewness)

## 9. Feature Engineering

Countries outside the 10 most common countries are grouped into **Other**. This reduces the number of categories before one-hot encoding.

In [ ]:
# Find the 10 most common countries
top_countries = df["Country"].value_counts().index[:10].tolist()

print("Top 10 countries:")
print(top_countries)

In [ ]:
# Group less common countries into "Other"
def group_country(country):
    if country in top_countries:
        return country
    return "Other"

df["Grouped_country"] = df["Country"].apply(group_country)

print(df["Grouped_country"].value_counts())

## 10. Train-Test Split and Feature Selection

In [ ]:
from sklearn.model_selection import train_test_split

# Study_Hours is treated as a skewed numerical feature
skewed_columns = ["Study_Hours"]

# Other numerical features
other_numeric_columns = [
    "Age",
    "Avg_Daily_Usage_Hours",
    "Daily_Unlocks",
    "Physical_Activity_Hours",
    "Sleep_Hours_Per_Night"
]

# Stress Level has a natural order
ordinal_columns = ["Stress_Level"]

# Categorical features without a natural order
nominal_columns = [
    "Gender",
    "Academic_Level",
    "Most_Used_Platform",
    "Purpose_Of_Use",
    "Grouped_country"
]

feature_columns = (
    skewed_columns
    + other_numeric_columns
    + ordinal_columns
    + nominal_columns
)

X = df[feature_columns]
y = df["Mental_Health_Score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

## 11. Preprocessing

Different types of features need different preprocessing:

- **Study Hours:** log transformation + scaling
- **Other numerical features:** standard scaling
- **Stress Level:** ordinal encoding
- **Other categorical features:** one-hot encoding

A `ColumnTransformer` combines all of these preprocessing steps.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    StandardScaler,
    OrdinalEncoder,
    OneHotEncoder
)
from sklearn.compose import ColumnTransformer

# 1. Skewed numerical features
skewed_pipeline = Pipeline(steps=[
    ("log_transform", FunctionTransformer(np.log1p)),
    ("scale", StandardScaler())
])

# 2. Other numerical features
numeric_pipeline = Pipeline(steps=[
    ("scale", StandardScaler())
])

# 3. Ordinal feature
ordinal_pipeline = Pipeline(steps=[
    (
        "encode",
        OrdinalEncoder(
            categories=[["Low", "Medium", "High", "Very High"]]
        )
    )
])

# 4. Nominal categorical features
nominal_pipeline = Pipeline(steps=[
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

# Combine all preprocessing pipelines
preprocessor = ColumnTransformer(transformers=[
    ("skewed", skewed_pipeline, skewed_columns),
    ("numeric", numeric_pipeline, other_numeric_columns),
    ("ordinal", ordinal_pipeline, ordinal_columns),
    ("nominal", nominal_pipeline, nominal_columns)
])

## 12. Model Building

### 12.1 Baseline Model — Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

linear_regression_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train the model
linear_regression_pipeline.fit(X_train, y_train)

# Predictions
lr_predictions = linear_regression_pipeline.predict(X_test)
lr_train_predictions = linear_regression_pipeline.predict(X_train)

# Evaluation
lr_r2_test = r2_score(y_test, lr_predictions)
lr_r2_train = r2_score(y_train, lr_train_predictions)
lr_mae = mean_absolute_error(y_test, lr_predictions)

print(f"Training R2: {lr_r2_train:.4f}")
print(f"Testing R2:  {lr_r2_test:.4f}")
print(f"MAE:         {lr_mae:.4f}")

### 12.2 Random Forest — Default Settings

In [ ]:
from sklearn.ensemble import RandomForestRegressor

random_forest_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("random_forest", RandomForestRegressor(random_state=42))
])

# Train the model
random_forest_pipeline.fit(X_train, y_train)

# Predictions
rf_predictions = random_forest_pipeline.predict(X_test)
rf_train_predictions = random_forest_pipeline.predict(X_train)

# Evaluation
rf_r2_test = r2_score(y_test, rf_predictions)
rf_r2_train = r2_score(y_train, rf_train_predictions)
rf_mae = mean_absolute_error(y_test, rf_predictions)

print(f"Training R2: {rf_r2_train:.4f}")
print(f"Testing R2:  {rf_r2_test:.4f}")
print(f"MAE:         {rf_mae:.4f}")

### 12.3 Random Forest Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

parameter_grid = {
    "random_forest__n_estimators": [100, 200, 300],
    "random_forest__max_depth": [5, 10, 15],
    "random_forest__min_samples_split": [2, 5, 10],
    "random_forest__min_samples_leaf": [1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,
    param_distributions=parameter_grid,
    n_iter=15,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best parameters:")
print(random_search.best_params_)

In [ ]:
# Evaluate the tuned Random Forest
best_rf_pipeline = random_search.best_estimator_

rf_tuned_predictions = best_rf_pipeline.predict(X_test)

rf_tuned_r2 = r2_score(y_test, rf_tuned_predictions)
rf_tuned_mae = mean_absolute_error(y_test, rf_tuned_predictions)

print(f"Tuned Random Forest R2:  {rf_tuned_r2:.4f}")
print(f"Tuned Random Forest MAE: {rf_tuned_mae:.4f}")

## 13. Model Evaluation and Comparison

In [ ]:
# RMSE for each model
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_predictions))
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_tuned_rmse = np.sqrt(
    mean_squared_error(y_test, rf_tuned_predictions)
)

# Training R2 for tuned Random Forest
rf_tuned_train_predictions = best_rf_pipeline.predict(X_train)
rf_tuned_r2_train = r2_score(
    y_train,
    rf_tuned_train_predictions
)

# Compare all models
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest (default)",
        "Random Forest (tuned)"
    ],
    "Testing R2": [
        lr_r2_test,
        rf_r2_test,
        rf_tuned_r2
    ],
    "Training R2": [
        lr_r2_train,
        rf_r2_train,
        rf_tuned_r2_train
    ],
    "MAE": [
        lr_mae,
        rf_mae,
        rf_tuned_mae
    ],
    "RMSE": [
        lr_rmse,
        rf_rmse,
        rf_tuned_rmse
    ]
})

results

### How to read the metrics

- **R2:** Higher is generally better.
- **MAE:** Lower is better.
- **RMSE:** Lower is better.
- Compare training and testing R2 to check whether the model may be overfitting.

## 14. Save the Model

In [ ]:
import joblib

# Save the tuned Random Forest pipeline
joblib.dump(best_rf_pipeline, "Mental_Health_Model.pkl")

print("Model saved as Mental_Health_Model.pkl")

## 15. Final Summary

The notebook:
- explored the Mental Health Score dataset,
- checked data quality and outliers,
- cleaned duplicate and invalid physical-activity values,
- grouped rare countries,
- applied appropriate preprocessing to numerical and categorical features,
- trained Linear Regression and Random Forest models,
- tuned the Random Forest hyperparameters,
- compared model performance using R2, MAE, and RMSE,
- saved the tuned Random Forest pipeline for later use.